##### This notebook simulates an enterprise production environment for executing SQL queries.

##### To achieve this, the original dataset `WA_Fn-UseC_-Telco-Customer-Churn.csv` was split and loaded into separate relational tables within a PostgreSQL database, from which SQL queries were used to:
1. retrieve the complete raw dataset;
2. reperform some graphical analyses from EDA step, but in tabular format and directly using SQL queries.

# *Setup*

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

# Creating the Database and Populating it with the Data
To prepare the simulation environment for queries

In [9]:
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')

demographic_cols = ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents']
service_cols = ['customerID', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                              'TechSupport', 'StreamingTV', 'StreamingMovies']
account_cols = ['customerID', 'tenure', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

display(
    df.loc[:, demographic_cols].head(),
    df.loc[:, service_cols].head(),
    df.loc[:, account_cols].head()
)

,customerID,gender,SeniorCitizen,Partner,Dependents
0,7590-VHVEG,Female,0,Yes,No
1,5575-GNVDE,Male,0,No,No
2,3668-QPYBK,Male,0,No,No
3,7795-CFOCW,Male,0,No,No
4,9237-HQITU,Female,0,No,No


,customerID,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies
0,7590-VHVEG,No,No phone service,DSL,No,Yes,No,No,No,No
1,5575-GNVDE,Yes,No,DSL,Yes,No,Yes,No,No,No
2,3668-QPYBK,Yes,No,DSL,Yes,Yes,No,No,No,No
3,7795-CFOCW,No,No phone service,DSL,Yes,No,Yes,Yes,No,No
4,9237-HQITU,Yes,No,Fiber optic,No,No,No,No,No,No


,customerID,tenure,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,1,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,34,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,2,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,45,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,2,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [10]:
load_dotenv()
db_url = os.getenv('DATABASE_URL')
engine = create_engine(db_url)

In [11]:
%%time
df.loc[:, demographic_cols].to_sql(name='customers_demographic', con=engine, if_exists='replace', index=False)

df.loc[:, service_cols].to_sql(name='customers_services', con=engine, if_exists='replace', index=False)

df.loc[:, account_cols].to_sql(name='customers_account_info', con=engine, if_exists='replace', index=False)

CPU times: total: 984 ms
Wall time: 6.78 s


43

# Executing SQL Queries

In [71]:
load_dotenv()
db_url = os.getenv('DATABASE_URL')
engine = create_engine(db_url)

## Building (retrieving) the complete raw dataset

In [72]:
with open('../sql/query_retrieving-complete-dataset.sql', 'r') as query:
    query = query.read()
    df = pd.read_sql_query(sql=query, con=engine)
display(df.head(10))
display(df.info())

,customerID,gender,SeniorCitizen,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,tenure,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,No,No phone service,DSL,No,Yes,No,1,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,Yes,No,DSL,Yes,No,Yes,34,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,Yes,No,DSL,Yes,Yes,No,2,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,No,No phone service,DSL,Yes,No,Yes,45,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,Yes,No,Fiber optic,No,No,No,2,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
5,9305-CDSKC,Female,0,No,No,Yes,Yes,Fiber optic,No,No,Yes,8,Month-to-month,Yes,Electronic check,99.65,820.5,Yes
6,1452-KIOVK,Male,0,No,Yes,Yes,Yes,Fiber optic,No,Yes,No,22,Month-to-month,Yes,Credit card (automatic),89.10,1949.4,No
7,6713-OKOMC,Female,0,No,No,No,No phone service,DSL,Yes,No,No,10,Month-to-month,No,Mailed check,29.75,301.9,No
8,7892-POOKP,Female,0,Yes,No,Yes,Yes,Fiber optic,No,No,Yes,28,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes
9,6388-TABGU,Male,0,No,Yes,Yes,No,DSL,Yes,Yes,No,62,One year,No,Bank transfer (automatic),56.15,3487.95,No


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  tenure            7043 non-null   int64  
 12  Contract          7043 non-null   object 
 13  PaperlessBilling  7043 non-null   object 
 14  PaymentMethod     7043 non-null   object 
 15  MonthlyCharges    7043 non-null   float64
 16  TotalCharges      7043 non-null   object 


None

## Tabular analysis with SQL
Rerunning some EDA graphical analyses directly with SQL in tabular format

In [66]:
with open('../sql/query_churn-distribution.sql', 'r') as query:
    query = query.read()
    df = pd.read_sql_query(sql=query, con=engine)
display(df)

,Churn,Frequency
0,No,5174
1,Yes,1869


In [67]:
with open('../sql/query_contract-distribution.sql', 'r') as query:
    query = query.read()
    df = pd.read_sql_query(sql=query, con=engine)
display(df)

,Contract,Freq_General,Freq_Churn-No,Freq_Churn-Yes,Prop_Churn-No,Prop_Churn-Year
0,Month-to-month,3875,2220,1655,57.290323,42.709677
1,One year,1473,1307,166,88.730482,11.269518
2,Two year,1695,1647,48,97.168142,2.831858
